# Error Pattern Analysis
> Characterizing GPT-2's Failure Modes on SCAN, and Comparison to T5

**Motivation:** Prior work (preliminary T5 analysis) observed that T5 
fails on SCAN by generating an incorrect *number* of repeated actions, 
consistently across splits. This notebook tests whether the small 
GPT-2 model organism trained in this project exhibits the same 
failure signature, before any mechanistic claims about *why* it fails 
are built on top of that assumption.

**Core Question:** Is GPT-2's dominant failure mode on SCAN the same 
repetition-counting error observed in T5, or does it fail in a 
qualitatively different way?

# 1. Preparation

In [48]:
import numpy as np
import pandas as pd
import re
import json
from collections import Counter
from itertools import groupby

import sys, os
root_path = os.path.abspath("..")
src_path = os.path.join(root_path, "src")
for path in [root_path, src_path]:
    if path not in sys.path:
        sys.path.append(root_path)

from config.config import Config

In [49]:
def parse_failure_log(filepath):
    """
    Parses SCAN failure logs and returns a list of dictionaries 
    with columns: splitname, command, target, pred.
    """
    # Regex breakdown:
    # \[(.*?)\] -> Captures everything inside the square brackets (splitname)
    # COMMAND:.*?IN:\s*(.*?)\s*OUT: -> Captures everything between IN: and OUT: (command)
    # TARGET:\s*(.*?)\s*\| -> Captures everything between TARGET: and the next | (target)
    # PRED:\s*(.*)$ -> Captures everything after PRED: to the end of the line (pred)
    pattern = re.compile(r"\[(.*?)\]\s*COMMAND:\s*(.*?)\s*\|\s*TARGET:\s*(.*?)\s*\|\s*PRED:\s*(.*)$")
    
    parsed_rows = []
    
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # Skip empty lines
                
            match = pattern.search(line)
            if match:
                splitname, command, target, pred = match.groups()
                
                parsed_rows.append({
                    "splitname": splitname,
                    "command": command.strip(),
                    "target": target.strip(),
                    "pred": pred.strip()
                })
                
    return parsed_rows

In [50]:
cfg  = Config()
seeds = cfg.seeds

for seed in seeds:
    simple_data = parse_failure_log(f"../results/failure_cases/failure_cases_simple_seed{seed}.txt")
    length_data = parse_failure_log(f"../results/failure_cases/failure_cases_length_seed{seed}.txt")
    addprim_jump_data = parse_failure_log(f"../results/failure_cases/failure_cases_addprim_jump_seed{seed}.txt")

    globals()[f"simple_data_seed{seed}"] = simple_data
    globals()[f"length_data_seed{seed}"] = length_data
    globals()[f"addprim_jump_data_seed{seed}"] = addprim_jump_data

    globals()[f"all_failures_seed{seed}"] = pd.DataFrame(simple_data + length_data + addprim_jump_data)
    globals()[f"simple_failures_seed{seed}"] = pd.DataFrame(simple_data)
    globals()[f"length_failures_seed{seed}"] = pd.DataFrame(length_data)
    globals()[f"addprim_jump_failures_seed{seed}"] = pd.DataFrame(addprim_jump_data)

# 2.  Error Analysis Framework

In [133]:
# ========= GRAMMAR-BASED SUB-CLAUSE LENGTH COMPUTATION =========

REPEAT_MULTIPLIERS = {"twice": 2, "thrice": 3}
SPATIAL_MODIFIERS = {"opposite", "around"}
DIRECTIONS = {"left", "right"}
TURN_VERB = "turn"
MOVEMENT_VERBS = {"walk", "look", "run", "jump"}

def parse_subclause_details(text: str) -> dict:
    """
    Extract verb, direction, spatial modifier, and repeat multiplier
    from a single SCAN sub-clause's text.
    """
    tokens = text.split()
    repeat_mult = 1
    for tok in tokens:
        if tok in REPEAT_MULTIPLIERS:
            repeat_mult = REPEAT_MULTIPLIERS[tok]

    spatial = next((t for t in tokens if t in SPATIAL_MODIFIERS), None)
    direction = next((t for t in tokens if t in DIRECTIONS), None)
    verb = next((t for t in tokens if t in (MOVEMENT_VERBS | {TURN_VERB})), None)

    return {
        "text": text,
        "verb": verb,
        "direction": direction,
        "spatial": spatial,
        "repeat_mult": repeat_mult,
        "modifier": spatial or (
            "twice" if repeat_mult == 2 else "thrice" if repeat_mult == 3 else None
        ),
    }


def compute_base_length(verb: str, direction: str | None, spatial: str | None) -> int:
    """
    Compute the un-repeated action-token length of a sub-clause,
    per SCAN's compositional grammar. Validated by hand against
    three real examples (turn+opposite, jump+around, walk+opposite)
    before use — see conversation notes.
    """
    if verb == TURN_VERB:
        if spatial == "around":
            return 4
        elif spatial == "opposite":
            return 2
        else:
            return 1
    else:  # movement verb
        if spatial == "around":
            return 4
        elif spatial == "opposite":
            return 3
        elif direction is not None:
            return 2
        else:
            return 1


# def compute_subclause_length(details: dict) -> int:
#     base = compute_base_length(details["verb"], details["direction"], details["spatial"])
#     return base * details["repeat_mult"]

def compute_base_length(verb: str, direction: str | None, spatial: str | None) -> int:
    """
    Derived from get_atomic_unit_and_count's per-unit length x implied
    single-repeat count, to avoid maintaining grammar logic in two
    places independently.
    """
    fake_details = {"verb": verb, "direction": direction, "spatial": spatial, "repeat_mult": 1}
    unit, reps = get_atomic_unit_and_count(fake_details)
    return len(unit) * reps

def get_subclause_lengths(command: str) -> tuple[list[dict], list[int]]:
    """
    Parse a full command into sub-clauses and compute each one's
    expected target-token length.

    Returns
    -------
    tuple[list[dict], list[int]]
        (subclause_details, subclause_lengths) — same order as they
        appear in the command text. Note: "after" reverses EXECUTION
        order relative to command order, but we keep command-text
        order here; slicing logic downstream must account for this.
    """
    inner = command.replace("<sos>", "").replace("IN:", "").replace("OUT:", "").strip()
    parts = re.split(r"\s+(and|after)\s+", inner)
    clause_texts = parts[0::2]
    connectors = parts[1::2]

    details_list = [parse_subclause_details(t) for t in clause_texts]
    lengths = [compute_subclause_length(d) for d in details_list]

    # "after" means the SECOND command-text clause executes FIRST.
    # Reorder to execution order if "after" is used (assumes single
    # binary connector, consistent with SCAN's grammar depth).
    if connectors and connectors[0] == "after":
        details_list = details_list[::-1]
        lengths = lengths[::-1]

    return details_list, lengths

In [135]:
VERB_TOKEN = {"walk": "I_WALK", "look": "I_LOOK", "run": "I_RUN", "jump": "I_JUMP"}

def get_atomic_unit_and_count(details: dict) -> tuple[tuple[str, ...], int]:
    """
    Derive the atomic repeating unit and expected repeat count for a
    sub-clause, from its verb/direction/spatial-modifier/repeat_mult.
    Validated to reproduce compute_base_length's totals exactly.
    """
    verb = details["verb"]
    direction = details["direction"]
    spatial = details["spatial"]
    mult = details["repeat_mult"]
    dir_token = f"I_TURN_{direction.upper()}" if direction else None

    if verb == TURN_VERB:
        if spatial == "around":
            return (dir_token,), 4 * mult
        elif spatial == "opposite":
            return (dir_token,), 2 * mult
        else:
            return (dir_token,), 1 * mult
    else:
        action_token = VERB_TOKEN[verb]
        if spatial == "around":
            return (dir_token, action_token), 4 * mult
        elif spatial == "opposite":
            return (dir_token, dir_token, action_token), 1 * mult
        elif direction is not None:
            return (dir_token, action_token), 1 * mult
        else:
            return (action_token,), 1 * mult


def merge_same_unit_regions(details_list: list[dict]) -> list[dict]:
    """
    Merge adjacent sub-clauses (in execution order) that share an
    identical atomic unit into one region with summed expected reps,
    since their boundary is structurally invisible in the flattened
    token stream (e.g. "run right twice after run around right").
    """
    regions = []
    for details in details_list:
        unit, reps = get_atomic_unit_and_count(details)
        if regions and regions[-1]["unit"] == unit:
            regions[-1]["expected_reps"] += reps
            regions[-1]["modifiers"].append(details["modifier"])
        else:
            regions.append({"unit": unit, "expected_reps": reps, "modifiers": [details["modifier"]]})
    return regions


def count_unit_reps(tokens: list[str], start: int, unit: tuple[str, ...]) -> tuple[int, str, int]:
    """
    Count consecutive occurrences of `unit` in `tokens` starting at
    `start`. Returns (reps_found, stop_reason, next_ptr), where
    stop_reason is "pred_exhausted" (ran out of tokens) or
    "mismatch" (a token present but doesn't continue the unit).
    """
    ptr = start
    reps = 0
    n = len(tokens)
    ulen = len(unit)

    while True:
        if ptr + ulen > n:
            return reps, "pred_exhausted", ptr
        if tuple(tokens[ptr:ptr + ulen]) == unit:
            reps += 1
            ptr += ulen
        else:
            return reps, "mismatch", ptr

In [145]:
def classify_failure_per_clause(
    command: str, pred_tokens: list[str], target_tokens: list[str]
) -> dict:
    details_list, lengths = get_subclause_lengths(command)

    if sum(lengths) != len(target_tokens):
        return {
            "overall_label": "PARSE_MISMATCH", "first_error_clause_idx": None,
            "first_error_modifier": None, "length_check_passed": False,
            "clause_diagnostics": [],
        }

    regions = merge_same_unit_regions(details_list)
    pred_ptr = 0
    diagnostics = []
    saw_overcount = False
    all_prior_exact = True  # becomes False after ANY non-exact region

    for region_idx, region in enumerate(regions):
        unit = region["unit"]
        expected = region["expected_reps"]
        is_last_region = region_idx == len(regions) - 1

        reps_found, reason, new_ptr = count_unit_reps(pred_tokens, pred_ptr, unit)

        if reps_found == expected:
            diagnostics.append({"region_idx": region_idx, "status": "match", "reps_found": reps_found})
            pred_ptr = new_ptr
            continue

        if reps_found > expected:
            diagnostics.append({
                "region_idx": region_idx, "status": "overcount_continue",
                "reps_found": reps_found, "expected_reps": expected, "reason": reason,
            })
            saw_overcount = True
            all_prior_exact = False
            pred_ptr = new_ptr
            continue

        # reps_found < expected — fatal, decide label
        if reps_found == 0:
            if reason == "pred_exhausted" and all_prior_exact:
                label = "C_clause_omission"
            else:
                # Either wrong content (mismatch), or an earlier region
                # already wasn't exact — not a clean prefix omission
                label = "B_semantic"
        elif is_last_region:
            label = "A1_repetition_count"
        else:
            # Partial undercount, not the last region — always B,
            # regardless of reason (fixes case 17 generally)
            label = "B_semantic"

        diagnostics.append({
            "region_idx": region_idx, "status": label,
            "reps_found": reps_found, "expected_reps": expected, "reason": reason,
        })
        return {
            "overall_label": label, "first_error_clause_idx": region_idx,
            "first_error_modifier": region["modifiers"][0] if region["modifiers"] else None,
            "length_check_passed": True, "clause_diagnostics": diagnostics,
        }

    if saw_overcount:
        overall = "A1_repetition_count"
    elif pred_ptr < len(pred_tokens):
        overall = "D_over_generation"
    elif pred_ptr == len(pred_tokens):
        overall = "NO_ERROR_DETECTED"
    else:
        overall = "E_residual"

    return {
        "overall_label": overall, "first_error_clause_idx": None,
        "first_error_modifier": None, "length_check_passed": True,
        "clause_diagnostics": diagnostics,
    }

### 2.1 Smoke Test

In [147]:
failure_qa_df = pd.DataFrame([
    {
        "command": "<sos> IN: turn opposite right after look twice OUT:",
        "target": "I_LOOK I_LOOK I_TURN_RIGHT I_TURN_RIGHT",
        "pred": "I_LOOK I_LOOK I_LOOK I_TURN_RIGHT I_TURN_RIGHT",
        "error": "A1_repetition_count",
    },
    {
        "command": "<sos> IN: walk twice and turn right thrice OUT:",
        "target": "I_WALK I_WALK I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT",
        "pred": "I_WALK I_WALK I_WALK I_TURN_RIGHT I_TURN_RIGHT",
        "error": "A1_repetition_count",
    },
    {
        "command": "<sos> IN: jump around right thrice after jump around left twice OUT:",
        "target": "I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP",
        "pred": "I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP",
        "error": "A1_repetition_count",
    },
    {
        "command": "<sos> IN: run around left and jump left OUT:",
        "target": "I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_JUMP",
        "pred": "I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN",
        "error": "C_clause_omission",
    },
    {
        "command": "<sos> IN: look and jump right OUT:",
        "target": "I_LOOK I_TURN_RIGHT I_JUMP",
        "pred": "I_LOOK I_LOOK I_TURN_RIGHT I_JUMP",
        "error": "A1_repetition_count",
    },
    {
        "command": "<sos> IN: look opposite left and turn opposite left twice OUT:",
        "target": "I_TURN_LEFT I_TURN_LEFT I_LOOK I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT",
        "pred": "I_TURN_LEFT I_TURN_LEFT I_LOOK I_TURN_LEFT I_TURN_LEFT I_LOOK I_TURN_LEFT",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: run right twice after run around right OUT:",
        "target": "I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN",
        "pred": "I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN",
        "error": "A1_repetition_count",
    },
    {
        "command": "<sos> IN: run right thrice after turn around left OUT:",
        "target": "I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN",
        "pred": "I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_RUN I_TURN_RIGHT I_RUN",
        "error": "A1_repetition_count",
    },
    {
        "command": "<sos> IN: look right thrice and walk twice OUT:",
        "target": "I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_WALK I_WALK",
        "pred": "I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK",
        "error": "C_clause_omission",
    },
    {
        "command": "<sos> IN: turn around left twice and jump around right twice OUT:",
        "target": "I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP",
        "pred": "I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP",
        "error": "A1_repetition_count",
    },
    {
        "command": "<sos> IN: turn around right twice and run around left twice OUT:",
        "target": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN",
        "pred": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: run after run around left thrice OUT:",
        "target": "I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_RUN",
        "pred": "I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN",
        "error": "C_clause_omission",
    },
    {
        "command": "<sos> IN: jump around left thrice and run OUT:",
        "target": "I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_RUN",
        "pred": "I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP",
        "error": "C_clause_omission",
    },

    {
        "command": "<sos> IN: turn opposite right thrice and jump right thrice OUT:",
        "target": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP",
        "pred": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP",
        "error": "A1_repetition_count",  
    },
    {
        "command": "<sos> IN: walk opposite right after turn opposite right OUT:",
        "target": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_WALK",
        "pred": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT",
        "error": "B_semantic",
    },
    {
        "command": "<sos> IN: look around right thrice and jump right OUT:",
        "target": "I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_JUMP",
        "pred": "I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK",
        "error": "B_semantic",
    },
    {
        "command": "<sos> IN: walk around right thrice and look opposite right thrice OUT:",
        "target": "I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_TURN_RIGHT I_LOOK",
        "pred": "I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: walk around left twice and look OUT:",
        "target": "I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_LOOK",
        "pred": "I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: jump around left and run opposite left thrice OUT:",
        "target": "I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_TURN_LEFT I_RUN I_TURN_LEFT I_TURN_LEFT I_RUN I_TURN_LEFT I_TURN_LEFT I_RUN",
        "pred": "I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_TURN_LEFT I_RUN",
        "error": "A1_repetition_count", 
    },
    {
        "command": "<sos> IN: walk right thrice and look twice OUT:",
        "target": "I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_LOOK I_LOOK",
        "pred": "I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_WALK",
        "error": "B_semantic",
    },
    {
        "command": "<sos> IN: run left after walk around left thrice OUT:",
        "target": "I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_RUN",
        "pred": "I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK",
        "error": "B_semantic",  
    },
    {
        "command": "<sos> IN: walk left and look opposite right OUT:",
        "target": "I_TURN_LEFT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_LOOK",
        "pred": "I_TURN_LEFT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK",
        "error": "B_semantic",  
    },
    {
        "command": "<sos> IN: jump left thrice and run opposite left OUT:",
        "target": "I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_TURN_LEFT I_RUN",
        "pred": "I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_TURN_LEFT I_RUN",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: run around left and walk opposite right thrice OUT:",
        "target": "I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_RIGHT I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_WALK",
        "pred": "I_TURN_LEFT I_TURN_LEFT I_RUN I_TURN_LEFT I_TURN_LEFT I_RUN I_TURN_LEFT I_TURN_LEFT I_RUN I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: turn opposite right thrice and run around right twice OUT:",
        "target": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN",
        "pred": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_RUN",
        "error": "A1_repetition_count", 
    },
    {
        "command": "<sos> IN: look opposite left and jump left twice OUT:",
        "target": "I_TURN_LEFT I_TURN_LEFT I_LOOK I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP",
        "pred": "I_TURN_LEFT I_TURN_LEFT I_LOOK I_TURN_LEFT I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP",
        "error": "A1_repetition_count",  
    },
    {
        "command": "<sos> IN: look around right twice and run around right twice OUT:",
        "target": "I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN",
        "pred": "I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK",
        "error": "B_semantic",  
    },
    {
        "command": "<sos> IN: look right twice and look opposite right twice OUT:",
        "target": "I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_TURN_RIGHT I_LOOK",
        "pred": "I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: walk around right twice after turn opposite right twice OUT:",
        "target": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK",
        "pred": "I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT",
        "error": "B_semantic",
    },
    {
        "command": "<sos> IN: turn right twice after walk right thrice OUT:",
        "target": "I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT",
        "pred": "I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_WALK",
        "error": "B_semantic",  
    },
    {
        "command": "<sos> IN: run left twice and walk around right OUT:",
        "target": "I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK",
        "pred": "I_TURN_LEFT I_WALK I_TURN_LEFT I_WALK I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN",
        "error": "B_semantic", 
    },
    {
        "command": "<sos> IN: jump around right twice and walk opposite left thrice OUT:",
        "target": "I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_RIGHT I_JUMP I_TURN_LEFT I_TURN_LEFT I_WALK I_TURN_LEFT I_TURN_LEFT I_WALK I_TURN_LEFT I_TURN_LEFT I_WALK",
        "pred": "I_TURN_RIGHT I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_WALK I_TURN_RIGHT I_TURN_RIGHT I_WALK I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP",
        "error": "B_semantic",  
    },
    {
        "command": "<sos> IN: run twice and turn left OUT:",
        "target": "I_RUN I_RUN I_TURN_LEFT",
        "pred": "I_RUN I_RUN I_TURN_LEFT I_RUN",
        "error": "D_over_generation", 
    },
])

print(f"Total QA examples: {len(failure_qa_df)}")

Total QA examples: 33


In [148]:
def validate_against_labels(labeled_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compare classify_failure_per_clause's output against hand-assigned
    ground-truth labels in labeled_df['error'].

    Parameters
    ----------
    labeled_df : pd.DataFrame
        Must have 'command', 'pred', 'target', 'error' columns.

    Returns
    -------
    pd.DataFrame
        Original rows plus 'predicted_type', 'match', and full
        per-clause diagnostics for any mismatches, for manual review.
    """
    df = labeled_df.copy()

    diagnostics = df.apply(
        lambda row: classify_failure_per_clause(
            row["command"], row["pred"].split(), row["target"].split()
        ),
        axis=1,
    )

    df["predicted_type"] = diagnostics.apply(lambda d: d["overall_label"])
    df["first_error_clause_idx"] = diagnostics.apply(lambda d: d["first_error_clause_idx"])
    df["first_error_modifier"] = diagnostics.apply(lambda d: d["first_error_modifier"])
    df["clause_diagnostics"] = diagnostics.apply(lambda d: d["clause_diagnostics"])

    df["match"] = df["error"] == df["predicted_type"]

    n_mismatch = (~df["match"]).sum()
    print(f"{len(df) - n_mismatch}/{len(df)} match.")

    if n_mismatch > 0:
        print(f"\n{n_mismatch} mismatch(es) — review each before trusting at scale:\n")
        mismatches = df[~df["match"]]
        for idx, row in mismatches.iterrows():
            print(f"[{idx}] COMMAND: {row['command']}")
            print(f"    Expected: {row['error']} | Got: {row['predicted_type']}")
            print(f"    First error clause: {row['first_error_clause_idx']} "
                  f"(modifier: {row['first_error_modifier']})")
            print(f"    Clause diagnostics: {row['clause_diagnostics']}")
            print()

    return df


validation_results = validate_against_labels(failure_qa_df)

27/33 match.

6 mismatch(es) — review each before trusting at scale:

[2] COMMAND: <sos> IN: jump around right thrice after jump around left twice OUT:
    Expected: A1_repetition_count | Got: PARSE_MISMATCH
    First error clause: nan (modifier: nan)
    Clause diagnostics: []

[5] COMMAND: <sos> IN: look opposite left and turn opposite left twice OUT:
    Expected: B_semantic | Got: A1_repetition_count
    First error clause: 1.0 (modifier: opposite)
    Clause diagnostics: [{'region_idx': 0, 'status': 'overcount_continue', 'reps_found': 2, 'expected_reps': 1, 'reason': 'pred_exhausted'}, {'region_idx': 1, 'status': 'A1_repetition_count', 'reps_found': 1, 'expected_reps': 4, 'reason': 'pred_exhausted'}]

[7] COMMAND: <sos> IN: run right thrice after turn around left OUT:
    Expected: A1_repetition_count | Got: B_semantic
    First error clause: 1.0 (modifier: thrice)
    Clause diagnostics: [{'region_idx': 0, 'status': 'overcount_continue', 'reps_found': 8, 'expected_reps': 4, 'reas

## 2.2 Full Split-Seed Implementation

In [157]:
os.makedirs("../results/failure_patterns", exist_ok=True)

def classify_all_failures_per_clause(failure_df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply classify_failure_per_clause to every row of a failure
    DataFrame. Expects 'command', 'pred', 'target' columns.

    Returns
    -------
    pd.DataFrame
        Copy of failure_df with added columns: 'error_type',
        'first_error_clause_idx', 'first_error_modifier',
        'length_check_passed'.
    """
    df = failure_df.copy()

    results = df.apply(
        lambda row: classify_failure_per_clause(
            row["command"], row["pred"].split(), row["target"].split()
        ),
        axis=1,
    )

    df["error_type"] = results.apply(lambda r: r["overall_label"])
    df["first_error_clause_idx"] = results.apply(lambda r: r["first_error_clause_idx"])
    df["first_error_modifier"] = results.apply(lambda r: r["first_error_modifier"])
    df["length_check_passed"] = results.apply(lambda r: r["length_check_passed"])

    return df


def classify_and_save_split(split: str, seeds: list[int]) -> None:
    """
    Classify all failure cases for a given split, across all seeds,
    using the per-clause grammar-aware classifier, and save
    per-example records (including modifier attribution) to a single
    split-specific JSON file.
    """
    records = []
    n_parse_mismatch = 0
    n_total = 0

    for seed in seeds:
        failure_df = globals()[f"{split}_failures_seed{seed}"]
        classified_df = classify_all_failures_per_clause(failure_df)

        n_total += len(classified_df)
        n_parse_mismatch += (classified_df["error_type"] == "PARSE_MISMATCH").sum()

        for _, row in classified_df.iterrows():
            records.append({
                "seed": seed,
                "split": split,
                "command": row["command"],
                "target": row["target"],
                "pred": row["pred"],
                "error_type": row["error_type"],
                "first_error_clause_idx": row["first_error_clause_idx"],
                "first_error_modifier": row["first_error_modifier"],
                "length_check_passed": row["length_check_passed"],
            })

    with open(f"../results/failure_patterns/{split}_error_classification.json", "w") as f:
        json.dump(records, f, indent=2)

    parse_mismatch_rate = n_parse_mismatch / n_total if n_total > 0 else 0.0
    print(f"Saved {len(records)} classified failure records for split={split}")
    print(f"  PARSE_MISMATCH rate: {parse_mismatch_rate:.2%} "
          f"({n_parse_mismatch}/{n_total})"
          + ("  <-- investigate before trusting proportions" if parse_mismatch_rate > 0.02 else ""))


for split in ["simple", "length", "addprim_jump"]:
    classify_and_save_split(split, seeds)

Saved 15668 classified failure records for split=simple
  PARSE_MISMATCH rate: 0.00% (0/15668)
Saved 13554 classified failure records for split=length
  PARSE_MISMATCH rate: 0.00% (0/13554)
Saved 27913 classified failure records for split=addprim_jump
  PARSE_MISMATCH rate: 0.00% (0/27913)


In [158]:
def aggregate_error_type_proportions(split: str, exclude_parse_mismatch: bool = True) -> pd.DataFrame:
    """
    Load a split's per-example classification records and compute
    per-seed error-type proportions, then aggregate (mean ± std)
    across seeds.

    Parameters
    ----------
    exclude_parse_mismatch : bool
        If True (default), PARSE_MISMATCH cases are dropped before
        computing proportions, so the denominator reflects only
        cases the grammar parser could actually handle. The count
        and rate of excluded cases is printed regardless, so this
        never happens silently.
    """
    with open(f"../results/failure_patterns/{split}_error_classification.json") as f:
        records = json.load(f)

    df = pd.DataFrame(records)

    n_mismatch = (df["error_type"] == "PARSE_MISMATCH").sum()
    if n_mismatch > 0:
        print(f"  [{split}] Excluding {n_mismatch}/{len(df)} PARSE_MISMATCH cases "
              f"({n_mismatch/len(df):.2%}) from proportions below.")

    if exclude_parse_mismatch:
        df = df[df["error_type"] != "PARSE_MISMATCH"]

    per_seed_props = (
        df.groupby(["seed", "error_type"])
        .size()
        .groupby(level=0)
        .apply(lambda x: x / x.sum())
        .unstack(fill_value=0.0)
    )

    summary = pd.DataFrame({
        "mean": per_seed_props.mean(),
        "std": per_seed_props.std(),
    })

    return summary


for split in ["simple", "length", "addprim_jump"]:
    print(f"\n{split.upper()} - error type proportions (mean ± std across seeds)")
    print(aggregate_error_type_proportions(split))


SIMPLE - error type proportions (mean ± std across seeds)
                         mean       std
error_type                             
A1_repetition_count  0.266339  0.096782
B_semantic           0.694267  0.109225
C_clause_omission    0.030601  0.012053
D_over_generation    0.008793  0.004338

LENGTH - error type proportions (mean ± std across seeds)
                         mean       std
error_type                             
A1_repetition_count  0.288609  0.126770
B_semantic           0.683380  0.134569
C_clause_omission    0.024814  0.009784
D_over_generation    0.003198  0.002292

ADDPRIM_JUMP - error type proportions (mean ± std across seeds)
                         mean       std
error_type                             
A1_repetition_count  0.244222  0.100435
B_semantic           0.720206  0.112492
C_clause_omission    0.029929  0.012278
D_over_generation    0.005644  0.003841


In [160]:
import json
import random

with open("../results/failure_patterns/simple_error_classification.json") as f:
    records = json.load(f)

b_examples = [r for r in records if r["error_type"] == "B_semantic"]
print(f"Total B examples: {len(b_examples)} / {len(records)}")

sample = random.sample(b_examples, min(20, len(b_examples)))
for ex in sample:
    print(f"COMMAND: {ex['command']}")
    print(f"TARGET:  {ex['target']}")
    print(f"PRED:    {ex['pred']}")
    print(f"first_error_clause_idx: {ex['first_error_clause_idx']}, modifier: {ex['first_error_modifier']}")
    print("-" * 60)

Total B examples: 10992 / 15668
COMMAND: <sos> IN: look opposite right after jump around left twice OUT:
TARGET:  I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_RIGHT I_TURN_RIGHT I_LOOK
PRED:    I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_LOOK
first_error_clause_idx: 1.0, modifier: opposite
------------------------------------------------------------
COMMAND: <sos> IN: walk opposite left twice and jump OUT:
TARGET:  I_TURN_LEFT I_TURN_LEFT I_WALK I_TURN_LEFT I_TURN_LEFT I_WALK I_JUMP
PRED:    I_TURN_LEFT I_TURN_LEFT I_TURN_LEFT I_JUMP I_WALK
first_error_clause_idx: 0.0, modifier: opposite
------------------------------------------------------------
COMMAND: <sos> IN: run opposite left and look right thrice OUT:
TARGET:  I_TURN_L

# 3. Semantic Error Deep Dive

For cases classified as semantic errors (B), we ask two follow-up 
questions:

**B1. Confusion Direction:** For each action type, is it typically 
under-generated (model avoids it when it should appear) or 
over-generated (model substitutes it in when it shouldn't)? Computed 
as the net difference between predicted and target counts per action, 
aggregated across all semantic-error cases. This is reported as a 
symmetric confusion tally, not a causal attribution — an 
under-generated action and an over-generated action in the same 
example are two views of the same substitution event, not 
independent causes.

**B2. Most-Confused Actions:** Which action types are involved in 
substitution errors most frequently, ranked by total confusion 
magnitude (sum of absolute net differences across all semantic-error 
cases)?



In [ ]:
# ========= SEMANTIC ERROR DEEP DIVE =========

def action_confusion(pred_tokens: list[str], target_tokens: list[str]) -> dict[str, int]:
    """
    Compute net difference (pred_count - target_count) per action.
    Negative = under-generated, Positive = over-generated.
    This quantifies discrepancy, not causal attribution.
    """
    pred_counts = Counter(pred_tokens)
    target_counts = Counter(target_tokens)
    all_actions = set(pred_counts) | set(target_counts)
    return {
        action: pred_counts.get(action, 0) - target_counts.get(action, 0)
        for action in all_actions
    }


def aggregate_semantic_confusion(semantic_error_df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate action confusion across all semantic-error cases.

    Returns
    -------
    pd.DataFrame
        Columns: action, net_under_generated, net_over_generated,
        total_confusion_magnitude. Sorted descending by magnitude.
    """
    under = Counter()
    over = Counter()

    for _, row in semantic_error_df.iterrows():
        diffs = action_confusion(row["pred"].split(), row["target"].split())
        for action, diff in diffs.items():
            if diff < 0:
                under[action] += -diff
            elif diff > 0:
                over[action] += diff

    all_actions = set(under) | set(over)
    records = [
        {
            "action": action,
            "net_under_generated": under.get(action, 0),
            "net_over_generated": over.get(action, 0),
            "total_confusion_magnitude": under.get(action, 0) + over.get(action, 0),
        }
        for action in all_actions
    ]
    return pd.DataFrame(records).sort_values(
        "total_confusion_magnitude", ascending=False
    ).reset_index(drop=True)

# 4. Syntactic Error Attribution to Modifiers

For cases classified as syntactic errors (A1 or A2), we ask which 
command modifier (*twice*, *thrice*, *around*, *after*, *opposite*) 
is associated with the error, when a single command may contain 
multiple modifiers across multiple sub-clauses.

**Method:** Each command is decomposed into its constituent 
sub-clauses (split on *and*/*after*), and the target action sequence 
is sliced into corresponding contiguous blocks using the known 
action-count grammar of each sub-clause type. The predicted sequence 
is compared against the target block-by-block, in order. The 
**first** sub-clause at which a mismatch occurs is attributed as the 
source of the error; all subsequent sub-clauses in that example are 
excluded from attribution, since an error at one position corrupts 
the autoregressive context for everything that follows, making later 
comparisons uninformative about which modifier is actually 
responsible.

**C1. Per-Modifier Error Rate:** For each modifier, 
`(# examples where this modifier's sub-clause is the first attributed 
error) / (# total appearances of this modifier across all sub-clauses, 
in both success and failure cases)`.

**C2. Modifier Co-occurrence Among Errors:** A Venn diagram showing, 
among all syntactic-error examples, which combinations of modifiers 
are *present in the command* (regardless of which one was actually 
attributed as the error source). This shows co-occurrence, not 
causation — e.g., if "twice" and "around" frequently co-occur in 
erroring commands, this does not imply both contributed to the 
error, only that commands combining them are disproportionately 
represented among failures.

In [ ]:
MODIFIERS = ["twice", "thrice", "around", "after", "opposite"]

def parse_subclauses(command: str) -> list[dict]:
    """
    Decompose a SCAN command into sub-clauses split on 'and'/'after'
    at the top level, extracting the modifier (if any) present in
    each sub-clause and the expected action-count multiplier.

    Parameters
    ----------
    command : str
        Full command string, e.g.
        "IN: turn opposite right thrice and run opposite left OUT:"

    Returns
    -------
    list[dict]
        One entry per sub-clause, each with keys:
        "text", "modifier" (str or None), "repeat_multiplier" (int)

    Note: this is a simplified grammar-aware parser tuned to SCAN's
    known compositional structure (base action + optional direction
    modifier + optional repeat modifier, joined by 'and'/'after').
    It should be validated against a sample of real commands before
    being trusted at scale — SCAN's grammar has edge cases (e.g.
    'after' reverses clause order semantically) that may need
    additional handling.
    """
    inner = command.replace("IN:", "").replace("OUT:", "").strip()

    # Split on top-level conjunctions; SCAN commands do not nest these
    parts = re.split(r"\s+(and|after)\s+", inner)
    clauses = parts[0::2]  # text chunks
    connectors = parts[1::2]  # "and" / "after" between them

    subclauses = []
    for clause_text in clauses:
        modifier = next((m for m in MODIFIERS if m in clause_text.split()), None)
        repeat_multiplier = 2 if "twice" in clause_text else (
            3 if "thrice" in clause_text else 1
        )
        subclauses.append({
            "text": clause_text.strip(),
            "modifier": modifier,
            "repeat_multiplier": repeat_multiplier,
        })

    return subclauses


def attribute_first_error_to_modifier(
    command: str,
    pred_tokens: list[str],
    target_tokens: list[str],
    subclause_action_lengths: list[int],
) -> str | None:
    """
    Attribute a syntactic error to the modifier of the FIRST sub-clause
    at which predicted and target action blocks diverge. Sub-clauses
    after the first divergence are not evaluated, since an error at
    one position corrupts the autoregressive context for everything
    downstream, making later comparisons uninformative.

    Parameters
    ----------
    command : str
        Full command string.
    pred_tokens, target_tokens : list[str]
        Predicted and target action sequences.
    subclause_action_lengths : list[int]
        Expected number of target action tokens contributed by each
        sub-clause, in order (must sum to len(target_tokens)).

    Returns
    -------
    str | None
        The modifier string attributed to the first error, or None if
        the first erroring sub-clause has no modifier, or if no
        sub-clause boundary could be aligned (parsing failure).
    """
    subclauses = parse_subclauses(command)

    if sum(subclause_action_lengths) != len(target_tokens):
        # Parsing/length mismatch — do not guess, flag as unattributable
        return None

    target_start = 0
    for subclause, length in zip(subclauses, subclause_action_lengths):
        target_block = target_tokens[target_start: target_start + length]
        pred_block = pred_tokens[target_start: target_start + length]

        if pred_block != target_block:
            return subclause["modifier"]  # may be None if this clause had none

        target_start += length

    return None  # no divergence found (should not occur if this IS a failure case)